In [29]:
from rag.wrappers import RagPipelineWrapper
import yaml
with open("config.yml", "r") as config_file:
    config = yaml.safe_load(config_file)
pipeline_file = f"{config["pipelines-dir"]}/{config["rag-demo"]["pipeline"]}"
mistral_file = "pipelines/mistral.yml"
chroma_path = config["vector-db"]["path"]
collection = config["vector-db"]["collection"]
prompt = config["rag-demo"]["prompt"]
llama3_1_pipe = RagPipelineWrapper(
    pipeline_file,
    chroma_path=chroma_path,
    collection=collection,
    prompt=prompt,
)
llama3_pipe = RagPipelineWrapper(
    "pipelines/llama3.yml",
    chroma_path=chroma_path,
    collection=collection,
    prompt=prompt,
)
mistral_pipe = RagPipelineWrapper(
    "pipelines/mistral.yml",
    chroma_path=chroma_path,
    collection=collection,
    prompt=prompt
)
pipes = {"mistral-nemo 12B": mistral_pipe, "llama3 8B": llama3_pipe, "llama3-1 8B": llama3_1_pipe}

In [31]:
import time
time_col = []
model_col = []
for query in config["rag-demo"]["examples"]:
    for key in pipes:
        for i in range(5):
            start = time.time()
            pipes[key].query(query)
            end = time.time()
            time_col.append(end-start)
            model_col.append(key)

In [32]:
import pandas as pd
import plotly.express as px
df = pd.DataFrame({"time":time_col,"model":model_col})
fig = px.box(df, x="time", y="model", color="model", points="all", title="RAG Benchmarks (RTX 4070 Super)", labels={"time": "Query Time (s)", "model": "LLM Model"})
fig.show()